In [1]:
from langchain_groq import ChatGroq
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

In [2]:
llm = ChatGroq(
        model="qwen/qwen3.8-27b", 
        
)

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

In [3]:
import json
from langchain_core.documents import Document

with open('policy.json','r') as f:
    data = json.load(f)
    
documents = []

for item in data:
    documents.append(
        Document(page_content=item['content'],
                 metadata={
                     'id':item['id'],
                     'title':item['title'],
                     'section':item['section'],
                     'source':'json'    
                 })
        
    )
    
documents
    

[Document(metadata={'id': 'P001', 'title': 'Data Privacy Policy', 'section': 'Retention', 'source': 'json'}, page_content='User data is retained for 90 days. It should not be retained after that.'),
 Document(metadata={'id': 'P002', 'title': 'Holiday Policy', 'section': 'Leave', 'source': 'json'}, page_content='Only 10 holidays should be given for any employee in a given year. It includes optional Holidays as well.'),
 Document(metadata={'id': 'P003', 'title': 'Earned Leave Policy', 'section': 'Leave', 'source': 'json'}, page_content="Each employee is eligible for 5 earned leaves per quarter, aligning with our commitment to supporting workâ€“life balance while ensuring smooth operational continuity. Earned leaves are designed to provide team members with the flexibility to pause, rest, or attend to personal commitments without compromising their professional responsibilities. These leaves may be availed at any time during the quarter, subject to prior approval from the reporting manage

In [4]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_metadata = {
        'description':'Policy documents indexed with metadata aware embeddings',
        'source type':'json'
    }
)



In [5]:
vectorstore_db = vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs = {
        'k':3,
        'filter':{'section':'Leave'},
        'lambda_mult':0.3,
    }
)

In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate([
    ('system',"""You are a policy assistant. Answer user questions strictly based on the provided context.
     Do not introduce external knowlege or assumptions. 
     <context>
     {context}
     </context> """),
    
    ('human','{input}')
])

print(prompt)

input_variables=['context', 'input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='You are a policy assistant. Answer user questions strictly based on the provided context.\n     Do not introduce external knowlege or assumptions. \n     <context>\n     {context}\n     </context> '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]


In [7]:
from langchain_core.prompts import PromptTemplate
document_prompt = PromptTemplate.from_template(
    """ 
    Source: {source}
    Policy ID: {id}
    Section: {section}
    Content: {page_content}
    """
)

In [8]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

docs_chain = create_stuff_documents_chain(
    llm=llm,
    prompt = prompt,
    document_prompt=document_prompt,
    document_variable_name='context'
)

rag_chain = create_retrieval_chain(
    vectorstore_db,
    docs_chain
)

In [9]:
print(rag_chain.invoke({"input":"what does policy says aboout unused earned leaves ?"}))


{'input': 'what does policy says aboout unused earned leaves ?', 'context': [Document(metadata={'title': 'Earned Leave Policy', 'section': 'Leave', 'id': 'P003', 'source': 'json'}, page_content="Each employee is eligible for 5 earned leaves per quarter, aligning with our commitment to supporting workâ€“life balance while ensuring smooth operational continuity. Earned leaves are designed to provide team members with the flexibility to pause, rest, or attend to personal commitments without compromising their professional responsibilities. These leaves may be availed at any time during the quarter, subject to prior approval from the reporting manager to ensure adequate workforce planning.For trainees, the entitlement is 4 earned leaves per quarter. This distinction recognizes the structured nature of training programs and the importance of consistent participation during the learning phase. Trainees are encouraged to utilize these leaves responsibly and plan ahead whenever possible. All e

In [11]:
print(rag_chain.invoke({"input":"what are the ids related to earned leave and holiday policy. Mention all.?"}))

{'input': 'what are the ids related to earned leave and holiday policy. Mention all.?', 'context': [Document(metadata={'title': 'Earned Leave Policy', 'section': 'Leave', 'id': 'P003', 'source': 'json'}, page_content="Each employee is eligible for 5 earned leaves per quarter, aligning with our commitment to supporting workâ€“life balance while ensuring smooth operational continuity. Earned leaves are designed to provide team members with the flexibility to pause, rest, or attend to personal commitments without compromising their professional responsibilities. These leaves may be availed at any time during the quarter, subject to prior approval from the reporting manager to ensure adequate workforce planning.For trainees, the entitlement is 4 earned leaves per quarter. This distinction recognizes the structured nature of training programs and the importance of consistent participation during the learning phase. Trainees are encouraged to utilize these leaves responsibly and plan ahead w